In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("Dataset1.csv")

In [4]:
df['Arrival_time'] = df['Arrival_time'].astype(str).str.strip()

In [5]:
df['Departure_Time'] = df['Departure_Time'].astype(str).str.strip()

In [6]:
df['Arrival_time'] = pd.to_datetime(
    df['Arrival_time'],
    format='%H:%M:%S',
    errors='coerce'
)

In [7]:
df['Departure_Time'] = pd.to_datetime(
    df['Departure_Time'],
    format='%H:%M:%S',
    errors='coerce'
)

In [8]:
print(df[['Arrival_time','Departure_Time']].head())

         Arrival_time      Departure_Time
0 1900-01-01 00:00:00 1900-01-01 10:25:00
1 1900-01-01 11:06:00 1900-01-01 11:08:00
2 1900-01-01 11:28:00 1900-01-01 11:30:00
3 1900-01-01 12:10:00 1900-01-01 00:00:00
4 1900-01-01 00:00:00 1900-01-01 20:30:00


In [9]:
print(df[['Arrival_time','Departure_Time']].isnull().sum())

Arrival_time      0
Departure_Time    0
dtype: int64


In [10]:
df_sorted = df.sort_values(['Train_No', 'SN'])

In [11]:
journey_duration = (
    df_sorted.groupby('Train_No')
    .agg(
        Start_Time=('Departure_Time', 'first'),
        End_Time=('Arrival_time', 'last')
    )
)

In [12]:
# Handle trains crossing midnight
journey_duration['Duration_Hours'] = (
    (journey_duration['End_Time'] - journey_duration['Start_Time'])
    .dt.total_seconds() / 3600
)

In [13]:
journey_duration.loc[
    journey_duration['Duration_Hours'] < 0,
    'Duration_Hours'
] += 24

In [14]:
journey_duration = journey_duration.reset_index()


In [15]:
print(journey_duration.head())

   Train_No          Start_Time            End_Time  Duration_Hours
0       107 1900-01-01 10:25:00 1900-01-01 12:10:00        1.750000
1       108 1900-01-01 20:30:00 1900-01-01 22:25:00        1.916667
2       128 1900-01-01 19:40:00 1900-01-01 17:45:00       22.083333
3       290 1900-01-01 18:30:00 1900-01-01 02:30:00        8.000000
4       401 1900-01-01 21:30:00 1900-01-01 10:00:00       12.500000


In [16]:
route_distance = (
    df.groupby('Train_No')['Distance']
      .max()
      .reset_index(name='Route_Distance')
)

In [17]:
def classify_route(distance):
    if distance < 300:
        return "Short"
    elif distance < 800:
        return "Medium"
    else:
        return "Long"

route_distance['Route_Type'] = (
    route_distance['Route_Distance']
    .apply(classify_route)
)

print(route_distance.head())

   Train_No  Route_Distance Route_Type
0       107              78      Short
1       108              83      Short
2       128             978       Long
3       290            2694       Long
4       401            1618       Long


In [18]:
print(route_distance['Route_Type'].value_counts())

Route_Type
Short     8221
Long      1546
Medium    1346
Name: count, dtype: int64


In [19]:
station_frequency = (
    df.groupby(['Station_Code', 'Station_Name'])
      .size()
      .reset_index(name='Train_Frequency')
      .sort_values('Train_Frequency', ascending=False)
)

print(station_frequency.head(10))

     Station_Code  Station_Name  Train_Frequency
1713         CSMT    CST-MUMBAI             1027
4254          KYN     KALYAN JN              828
7602          TNA         THANE              796
6649         SDAH       SEALDAH              745
5030          MSB  CHENNAI BEAC              738
2979          HWH    HOWRAH JN.              699
2130           DR         DADAR              567
1843          DDJ   DUM DUM JN.              463
1592          CLA         KURLA              462
7395          TBM      TAMBARAM              434


In [20]:
top10 = station_frequency.head(10)

print(top10)

     Station_Code  Station_Name  Train_Frequency
1713         CSMT    CST-MUMBAI             1027
4254          KYN     KALYAN JN              828
7602          TNA         THANE              796
6649         SDAH       SEALDAH              745
5030          MSB  CHENNAI BEAC              738
2979          HWH    HOWRAH JN.              699
2130           DR         DADAR              567
1843          DDJ   DUM DUM JN.              463
1592          CLA         KURLA              462
7395          TBM      TAMBARAM              434


In [21]:
print("===== LEVEL 2 SUMMARY =====")

print("\nRoute Categories:")
print(route_distance['Route_Type'].value_counts())

print("\nTop 10 Stations by Train Frequency:")
print(station_frequency.head(10))

print("\nSample Journey Durations:")
print(journey_duration.head())

===== LEVEL 2 SUMMARY =====

Route Categories:
Route_Type
Short     8221
Long      1546
Medium    1346
Name: count, dtype: int64

Top 10 Stations by Train Frequency:
     Station_Code  Station_Name  Train_Frequency
1713         CSMT    CST-MUMBAI             1027
4254          KYN     KALYAN JN              828
7602          TNA         THANE              796
6649         SDAH       SEALDAH              745
5030          MSB  CHENNAI BEAC              738
2979          HWH    HOWRAH JN.              699
2130           DR         DADAR              567
1843          DDJ   DUM DUM JN.              463
1592          CLA         KURLA              462
7395          TBM      TAMBARAM              434

Sample Journey Durations:
   Train_No          Start_Time            End_Time  Duration_Hours
0       107 1900-01-01 10:25:00 1900-01-01 12:10:00        1.750000
1       108 1900-01-01 20:30:00 1900-01-01 22:25:00        1.916667
2       128 1900-01-01 19:40:00 1900-01-01 17:45:00       22.083